# RouteRanker — Task 1: Explainable Route Ranking

**Project:** RouteRanker – AI-Powered Public Transit Route Optimization  
**Location:** Chennai, Tamil Nadu  
**Operator:** Metropolitan Transport Corporation (MTC), Chennai  
**Module:** `notebooks/07_explainable_route_ranking.ipynb`  

---

### Module Objective & Scope
This notebook implements an **Explainable Route Ranking Module** that ranks active Chennai MTC bus routes using verified GTFS schedule data and network topological characteristics.

**Core Methodological Principles:**
1. **GTFS Route Granularity:** Evaluated at the granular `route_id` level ($N = 4,611$).
2. **Active vs. Unserviced Segregation:**
   - Active routes ($N = 3,934$, possessing scheduled trips in `trips.txt`) are ranked from #1 to #3,934.
   - Unserviced routes ($N = 677$, zero scheduled trips) are quarantined with status `Unserviced` and excluded from active rankings.
3. **Additive Explainability:** Every route score ($0 - 100$) is the exact linear sum of individual criterion contributions, providing full transparency.
4. **No Demand Fabrication:** Passenger demand is not used as a ranking feature; rankings represent supply availability, headway convenience, and network connectivity.


In [1]:
import os
import sys
import re
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('ggplot')
import networkx as nx
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')

# Set working directories within mtc-gtfs
root = Path(r'C:\Users\kavyasri\Downloads\mtc-gtfs')
reports_dir = root / 'reports'
figures_dir = reports_dir / 'figures'
notebooks_dir = root / 'notebooks'

reports_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print("Environment initialized successfully.")
print(f"Project root: {root}")


Environment initialized successfully.
Project root: C:\Users\kavyasri\Downloads\mtc-gtfs


C:\Users\kavyasri\.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at C:\Users\kavyasri\AppData\Local\Temp\matplotlib-m6f_24ia because there was an issue with the default path (C:\Users\kavyasri\.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
# Load verified GTFS tables and project datasets
routes = pd.read_csv(root / 'routes.txt', dtype={'route_id': str, 'agency_id': str})
trips = pd.read_csv(root / 'trips.txt', dtype={'route_id': str, 'trip_id': str, 'service_id': str})
stops = pd.read_csv(root / 'stops.txt', dtype={'stop_id': str})
stop_times = pd.read_csv(root / 'stop_times.txt', dtype={'trip_id': str, 'stop_id': str})
calendar = pd.read_csv(root / 'calendar.txt', dtype={'service_id': str})
canonical = pd.read_csv(root / 'data' / 'canonical_routes_mapping.csv')
clust_assign = pd.read_csv(root / 'reports' / 'route_cluster_assignments.csv', dtype={'route_id': str})

# Granularity & Active Routes Audit
all_route_ids = set(routes['route_id'].unique())
active_route_ids = set(trips['route_id'].unique())
unserviced_route_ids = all_route_ids - active_route_ids

print("=== GTFS Dataset & Granularity Audit ===")
print(f"Total defined routes (routes.txt) : {len(all_route_ids):,}")
print(f"Active scheduled routes (trips.txt): {len(active_route_ids):,}")
print(f"Unserviced routes (no trips)       : {len(unserviced_route_ids):,}")
print(f"Total scheduled trips              : {len(trips):,}")
print(f"Total stops defined                : {len(stops):,}")
print(f"Total stop arrival/departure events: {len(stop_times):,}")


=== GTFS Dataset & Granularity Audit ===
Total defined routes (routes.txt) : 4,611
Active scheduled routes (trips.txt): 3,934
Unserviced routes (no trips)       : 677
Total scheduled trips              : 47,047
Total stops defined                : 5,580
Total stop arrival/departure events: 1,360,443


## 2. Multi-Criteria Feature Engineering

Six distinct supply-side operational criteria are engineered:
1. **`daily_scheduled_trips` (Frequency, Benefit $\uparrow$):** Average daily scheduled dispatches: $(5 \times \text{weekday} + 2 \times \text{weekend}) / 7$.
2. **`effective_headway_min` (Headway Convenience, Cost $\downarrow$):** Median minutes between consecutive departures. Clamped between $5\text{ min}$ (maximum cadence) and $180\text{ min}$ (infrequent/isolated trips).
3. **`stop_count` (Stop Coverage, Benefit $\uparrow$):** Total unique physical transit stops serviced.
4. **`avg_stop_degree` (Network Connectivity, Benefit $\uparrow$):** Average degree of route stops in the directed transit network graph ($G$: 5,477 nodes, 12,521 edges).
5. **`length_utility_km` (Corridor Reach Utility):** Route length (km) capped at $30\text{ km}$ to reward regional connectivity without favoring excessively circuitous lines.
6. **`service_span_hours` (Service Span, Benefit $\uparrow$):** Operating duration in hours between the first departure and last departure of the day.


In [3]:
# 1. Directed Transit Network Graph for Hub Connectivity
st_sorted = stop_times.sort_values(['trip_id', 'stop_sequence'])
mask = st_sorted['trip_id'] == st_sorted['trip_id'].shift(-1)
u_nodes = st_sorted['stop_id'][mask].values
v_nodes = st_sorted['stop_id'].shift(-1)[mask].values
edge_df = pd.DataFrame({'u': u_nodes, 'v': v_nodes}).drop_duplicates()
G = nx.from_pandas_edgelist(edge_df, source='u', target='v', create_using=nx.DiGraph())
deg_dict = dict(G.degree())

trip_stop = trips[['trip_id', 'route_id']].merge(stop_times[['trip_id', 'stop_id']], on='trip_id')[['route_id', 'stop_id']].drop_duplicates()
trip_stop['degree'] = trip_stop['stop_id'].map(deg_dict).fillna(1)
route_deg = trip_stop.groupby('route_id')['degree'].mean().reset_index().rename(columns={'degree': 'avg_stop_degree'})

# 2. Operating Service Span (hours)
def time_to_sec(t):
    try:
        parts = str(t).strip().split(':')
        return int(parts[0])*3600 + int(parts[1])*60 + int(parts[2])
    except:
        return np.nan

origin_times = stop_times.sort_values(['trip_id', 'stop_sequence']).groupby('trip_id').first().reset_index()
origin_times['dep_sec'] = origin_times['departure_time'].apply(time_to_sec)
trip_dep = trips[['trip_id', 'route_id']].merge(origin_times[['trip_id', 'dep_sec']], on='trip_id').dropna()
span = trip_dep.groupby('route_id')['dep_sec'].agg(['min', 'max']).reset_index()
span['service_span_hours'] = (span['max'] - span['min']) / 3600.0

# 3. Master DataFrame
master_df = routes[['route_id', 'route_short_name', 'route_long_name', 'route_type']].copy()
master_df = master_df.merge(clust_assign[['route_id', 'canonical_route', 'stop_count', 'route_length_km', 
                                          'weekday_trip_count', 'weekend_trip_count', 'weekday_headway_min', 
                                          'weekend_headway_min', 'kmeans_cluster']], on='route_id', how='left')
master_df = master_df.merge(route_deg, on='route_id', how='left')
master_df = master_df.merge(span[['route_id', 'service_span_hours']], on='route_id', how='left')

master_df['is_active'] = master_df['route_id'].isin(active_route_ids)

# Partition Active and Unserviced
active_df = master_df[master_df['is_active']].copy().reset_index(drop=True)
unserviced_df = master_df[~master_df['is_active']].copy().reset_index(drop=True)

# Derive exact active ranking metrics
active_df['daily_scheduled_trips'] = (active_df['weekday_trip_count'] * 5.0 + active_df['weekend_trip_count'] * 2.0) / 7.0
active_df['effective_headway_min'] = active_df['weekday_headway_min'].fillna(180.0).clip(lower=5.0, upper=180.0)
active_df['stop_count'] = active_df['stop_count'].fillna(2.0).astype(int)
active_df['route_length_km'] = active_df['route_length_km'].fillna(active_df['route_length_km'].median())
active_df['avg_stop_degree'] = active_df['avg_stop_degree'].fillna(active_df['avg_stop_degree'].median())
active_df['service_span_hours'] = active_df['service_span_hours'].fillna(0.0)

print(f"Feature engineering completed for {len(active_df):,} active routes.")
print(active_df[['daily_scheduled_trips', 'effective_headway_min', 'stop_count', 'route_length_km', 'service_span_hours', 'avg_stop_degree']].describe().round(2))


Feature engineering completed for 3,934 active routes.
       daily_scheduled_trips  ...  avg_stop_degree
count                3934.00  ...          3934.00
mean                   11.96  ...             9.33
std                    23.63  ...             3.30
min                     1.00  ...             1.78
25%                     2.00  ...             7.13
50%                     4.00  ...             9.12
75%                    10.00  ...            11.12
max                   210.00  ...            36.00

[8 rows x 6 columns]


## 3. Transparent Weighted Multi-Criteria Ranking

### Criteria Weights & Normalization:
$$\text{Overall\_Score}_i = 100 \times \sum_{j=1}^{6} w_j \cdot S_{ij}$$
$$\text{Contribution}_{ij} = 100 \times w_j \cdot S_{ij}$$

| Criterion ($j$) | Feature Column | Weight ($w_j$) | Scaling Function | Operational Rationale |
| :--- | :--- | :---: | :--- | :--- |
| **Trip Frequency** | `daily_scheduled_trips` | **25%** | Min-Max $[0, 1]$ | Quantifies total daily scheduled supply |
| **Headway Convenience** | `effective_headway_min` | **25%** | $(180 - H) / (180 - 5)$ | Rewards short waiting times; cost inverted |
| **Stop Coverage** | `stop_count` | **15%** | Min-Max $[0, 1]$ | Rewards spatial boarding access points |
| **Network Connectivity**| `avg_stop_degree` | **15%** | Min-Max $[0, 1]$ | Rewards connections to major interchange terminals |
| **Corridor Reach** | `route_length_km` | **10%** | $\min(30, L) / 30$ | Rewards regional connectivity up to 30 km |
| **Service Span** | `service_span_hours` | **10%** | $\min(18, \text{span}) / 18$ | Rewards dependable all-day service windows |


In [4]:
# Normalization
min_trips, max_trips = active_df['daily_scheduled_trips'].min(), active_df['daily_scheduled_trips'].max()
active_df['norm_frequency'] = (active_df['daily_scheduled_trips'] - min_trips) / (max_trips - min_trips)
active_df['norm_headway'] = (180.0 - active_df['effective_headway_min']) / (180.0 - 5.0)

min_stops, max_stops = active_df['stop_count'].min(), active_df['stop_count'].max()
active_df['norm_stop_coverage'] = (active_df['stop_count'] - min_stops) / (max_stops - min_stops)

min_deg, max_deg = active_df['avg_stop_degree'].min(), active_df['avg_stop_degree'].max()
active_df['norm_connectivity'] = (active_df['avg_stop_degree'] - min_deg) / (max_deg - min_deg)

active_df['norm_length_utility'] = active_df['route_length_km'].clip(upper=30.0) / 30.0
active_df['norm_service_span'] = active_df['service_span_hours'].clip(upper=18.0) / 18.0

# Criterion Weights
weights = {
    'frequency': 0.25,
    'headway': 0.25,
    'stop_coverage': 0.15,
    'connectivity': 0.15,
    'length_utility': 0.10,
    'service_span': 0.10
}

# Individual Feature Contributions (pts out of 100)
active_df['contrib_frequency'] = 100.0 * weights['frequency'] * active_df['norm_frequency']
active_df['contrib_headway'] = 100.0 * weights['headway'] * active_df['norm_headway']
active_df['contrib_stop_coverage'] = 100.0 * weights['stop_coverage'] * active_df['norm_stop_coverage']
active_df['contrib_connectivity'] = 100.0 * weights['connectivity'] * active_df['norm_connectivity']
active_df['contrib_length_utility'] = 100.0 * weights['length_utility'] * active_df['norm_length_utility']
active_df['contrib_service_span'] = 100.0 * weights['service_span'] * active_df['norm_service_span']

contrib_cols = [
    'contrib_frequency',
    'contrib_headway',
    'contrib_stop_coverage',
    'contrib_connectivity',
    'contrib_length_utility',
    'contrib_service_span'
]

# Overall Score & Ranking
active_df['overall_score'] = active_df[contrib_cols].sum(axis=1).round(2)
active_df = active_df.sort_values(by=['overall_score', 'daily_scheduled_trips'], ascending=[False, False]).reset_index(drop=True)
active_df['rank'] = range(1, len(active_df) + 1)
active_df['status'] = 'Active'

# Automated Explainability Generation
def generate_explanation(row):
    rank = int(row['rank'])
    score = row['overall_score']
    trips = row['daily_scheduled_trips']
    hw = row['effective_headway_min']
    stops = int(row['stop_count'])
    length = row['route_length_km']
    span = row['service_span_hours']
    c_freq = row['contrib_frequency']
    c_hw = row['contrib_headway']
    c_stops = row['contrib_stop_coverage']
    c_conn = row['contrib_connectivity']
    
    if rank <= 50:
        tier = "Premier High-Frequency Trunk Corridor"
        freq_desc = f"exceptional scheduled frequency ({trips:.0f} daily trips, +{c_freq:.1f} pts)"
        hw_desc = f"rapid departure cadence ({hw:.0f} min headway, +{c_hw:.1f} pts)"
        span_desc = f"extensive all-day service span ({span:.1f}h)"
    elif rank <= 300:
        tier = "Major Arterial Service Route"
        freq_desc = f"robust frequency ({trips:.0f} daily trips, +{c_freq:.1f} pts)"
        hw_desc = f"reliable headway ({hw:.0f} min, +{c_hw:.1f} pts)"
        span_desc = f"dependable operating span ({span:.1f}h)"
    elif rank <= 1500:
        tier = "Standard Feeder / Suburban Connector"
        freq_desc = f"moderate frequency ({trips:.0f} daily trips, +{c_freq:.1f} pts)"
        hw_desc = f"moderate headway ({hw:.0f} min, +{c_hw:.1f} pts)"
        span_desc = f"standard service window ({span:.1f}h)"
    else:
        tier = "Low-Frequency Short-Loop / Cut-Trip Variant"
        freq_desc = f"limited scheduled dispatches ({trips:.0f} daily trips, +{c_freq:.1f} pts)"
        hw_desc = f"infrequent service intervals ({hw:.0f} min, +{c_hw:.1f} pts)"
        span_desc = f"restricted operating window ({span:.1f}h)"
        
    expl = (f"{tier}. Ranked #{rank} with an overall score of {score:.1f}/100. "
            f"Driven by {freq_desc} and {hw_desc}. "
            f"Connects {stops} stops across {length:.1f} km with {span_desc} "
            f"(Coverage: +{c_stops:.1f} pts, Hub Connectivity: +{c_conn:.1f} pts).")
    return expl

active_df['score_explanation'] = active_df.apply(generate_explanation, axis=1)

# Format Unserviced Routes
unserviced_df['daily_scheduled_trips'] = 0.0
unserviced_df['effective_headway_min'] = np.nan
unserviced_df['service_span_hours'] = 0.0
for c in contrib_cols:
    unserviced_df[c] = 0.0
unserviced_df['overall_score'] = 0.0
unserviced_df['rank'] = np.nan
unserviced_df['status'] = 'Unserviced'
unserviced_df['score_explanation'] = "Unserviced route definition. Zero scheduled trips in GTFS timetable dispatches. Excluded from active ranking."

export_cols = [
    'rank', 'route_id', 'route_short_name', 'canonical_route', 'status',
    'overall_score', 'daily_scheduled_trips', 'effective_headway_min',
    'stop_count', 'route_length_km', 'service_span_hours', 'avg_stop_degree',
    'contrib_frequency', 'contrib_headway', 'contrib_stop_coverage',
    'contrib_connectivity', 'contrib_length_utility', 'contrib_service_span',
    'score_explanation'
]

full_results_df = pd.concat([active_df[export_cols], unserviced_df[export_cols]], ignore_index=True)
results_csv_path = reports_dir / 'route_ranking_results.csv'
full_results_df.to_csv(results_csv_path, index=False)

print(f"Exported ranking results ({len(full_results_df):,} routes) to: {results_csv_path}")
print(f"Active routes ranked: {len(active_df):,} (Ranks 1 to {len(active_df)})")
print(f"Unserviced routes quarantined: {len(unserviced_df):,}")


Exported ranking results (4,611 routes) to: C:\Users\kavyasri\Downloads\mtc-gtfs\reports\route_ranking_results.csv
Active routes ranked: 3,934 (Ranks 1 to 3934)
Unserviced routes quarantined: 677


In [5]:
# 4. Sensitivity Analysis across Policy Priorities
sensitivity_scenarios = {
    'Baseline (Balanced Policy)': weights,
    'Frequency-First (Capacity)': {
        'frequency': 0.45, 'headway': 0.30, 'stop_coverage': 0.10,
        'connectivity': 0.05, 'length_utility': 0.05, 'service_span': 0.05
    },
    'Coverage-First (Equity)': {
        'frequency': 0.15, 'headway': 0.15, 'stop_coverage': 0.30,
        'connectivity': 0.20, 'length_utility': 0.10, 'service_span': 0.10
    },
    'Equal-Weights (Benchmark)': {
        'frequency': 1/6, 'headway': 1/6, 'stop_coverage': 1/6,
        'connectivity': 1/6, 'length_utility': 1/6, 'service_span': 1/6
    }
}

sens_corr = {}
norm_cols_map = {
    'frequency': 'norm_frequency',
    'headway': 'norm_headway',
    'stop_coverage': 'norm_stop_coverage',
    'connectivity': 'norm_connectivity',
    'length_utility': 'norm_length_utility',
    'service_span': 'norm_service_span'
}

print("=== Spearman Rank Correlation Analysis ===")
for name, wdict in sensitivity_scenarios.items():
    s_score = sum(active_df[norm_cols_map[k]] * w * 100.0 for k, w in wdict.items())
    s_rank = s_score.rank(ascending=False)
    corr, _ = spearmanr(active_df['rank'], s_rank)
    sens_corr[name] = corr
    print(f"  - Baseline vs {name:<28}: rho = {corr:.4f}")


=== Spearman Rank Correlation Analysis ===
  - Baseline vs Baseline (Balanced Policy)  : rho = 1.0000
  - Baseline vs Frequency-First (Capacity)  : rho = 0.9832
  - Baseline vs Coverage-First (Equity)     : rho = 0.9214
  - Baseline vs Equal-Weights (Benchmark)   : rho = 0.9348


In [6]:
# 5. Visualizations Generation

# Figure 1: Score Distribution, Frequency vs Headway, Top 15 Routes, and Sensitivity
fig = plt.figure(figsize=(14, 10))

# Subplot 1: Score Distribution
ax1 = fig.add_subplot(2, 2, 1)
n, bins, patches = ax1.hist(active_df['overall_score'], bins=35, color='#2b5c8f', edgecolor='black', alpha=0.8)
ax1.axvline(active_df['overall_score'].median(), color='red', linestyle='--', linewidth=2, 
            label=f"Median: {active_df['overall_score'].median():.1f}")
ax1.axvline(active_df['overall_score'].quantile(0.90), color='green', linestyle=':', linewidth=2, 
            label=f"Top 10%: {active_df['overall_score'].quantile(0.90):.1f}")
ax1.set_title("A. Distribution of Overall Route Scores (Active, N=3,934)", fontsize=11, fontweight='bold')
ax1.set_xlabel("Overall Score (0 - 100)", fontsize=10)
ax1.set_ylabel("Number of Routes", fontsize=10)
ax1.legend(loc='upper right', fontsize=9)

# Subplot 2: Frequency vs Headway colored by Score
ax2 = fig.add_subplot(2, 2, 2)
sc = ax2.scatter(active_df['effective_headway_min'], active_df['daily_scheduled_trips'], 
                 c=active_df['overall_score'], cmap='viridis', s=25, alpha=0.7)
cbar = plt.colorbar(sc, ax=ax2)
cbar.set_label("Overall Score", fontsize=9)
ax2.set_title("B. Daily Frequency vs Headway", fontsize=11, fontweight='bold')
ax2.set_xlabel("Effective Headway (min)", fontsize=10)
ax2.set_ylabel("Daily Scheduled Trips", fontsize=10)

# Subplot 3: Top 15 Ranked Routes
ax3 = fig.add_subplot(2, 2, 3)
top15 = active_df.head(15).iloc[::-1]
y_pos = np.arange(len(top15))
labels = [f"#{int(r['rank'])}: {r['route_short_name']} ({r['route_id']})" for _, r in top15.iterrows()]
bars = ax3.barh(y_pos, top15['overall_score'], color='#1e7e34', edgecolor='black', height=0.7)
ax3.set_yticks(y_pos)
ax3.set_yticklabels(labels, fontsize=8.5)
ax3.set_title("C. Top 15 Ranked Chennai MTC Routes", fontsize=11, fontweight='bold')
ax3.set_xlabel("Overall Route Score", fontsize=10)
for bar in bars:
    ax3.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height()/2, f"{bar.get_width():.1f}", 
             va='center', fontsize=8, fontweight='bold')

# Subplot 4: Policy Sensitivity Correlations
ax4 = fig.add_subplot(2, 2, 4)
scen_names = list(sens_corr.keys())
scen_vals = [sens_corr[k] for k in scen_names]
bar_colors = ['#4a6fa5', '#166088', '#4fc1a1', '#e07a5f']
ax4.barh(scen_names[::-1], scen_vals[::-1], color=bar_colors[::-1], edgecolor='black', height=0.6)
ax4.set_xlim(0.85, 1.02)
ax4.set_title("D. Policy Weighting Sensitivity (Spearman rho)", fontsize=11, fontweight='bold')
ax4.set_xlabel("Correlation with Baseline Ranking", fontsize=10)
for i, v in enumerate(scen_vals[::-1]):
    ax4.text(v + 0.005, i, f"{v:.4f}", va='center', fontsize=9, fontweight='bold')

fig.tight_layout(pad=2.5)
fig_ranking_path = figures_dir / 'fig_route_ranking.png'
plt.savefig(fig_ranking_path, dpi=200)
plt.close()
print(f"Saved ranking overview figure to: {fig_ranking_path}")

# Figure 2: Stacked Feature Contributions for Top, Median, and Lower Routes
fig, ax = plt.subplots(figsize=(13, 7))
sample_top = active_df.head(8)
sample_mid = active_df.iloc[len(active_df)//2 - 2 : len(active_df)//2 + 2]
sample_bot = active_df.tail(4)
sample_routes = pd.concat([sample_top, sample_mid, sample_bot], ignore_index=True)

y_labels = [f"#{int(r['rank'])}: {r['route_short_name']} (Sc: {r['overall_score']:.1f})" for _, r in sample_routes.iterrows()]
y_indices = np.arange(len(sample_routes))

crit_names = ['contrib_frequency', 'contrib_headway', 'contrib_stop_coverage', 
              'contrib_connectivity', 'contrib_length_utility', 'contrib_service_span']
crit_labels = ['Trip Frequency (25%)', 'Headway Convenience (25%)', 'Stop Coverage (15%)',
               'Hub Connectivity (15%)', 'Corridor Length (10%)', 'Operating Span (10%)']
colors = ['#2a9d8f', '#e76f51', '#f4a261', '#457b9d', '#1d3557', '#a8dadc']

bottom = np.zeros(len(sample_routes))
for c_col, c_label, c_color in zip(crit_names, crit_labels, colors):
    vals = sample_routes[c_col].values
    ax.barh(y_indices[::-1], vals, left=bottom, label=c_label, color=c_color, edgecolor='white', height=0.65)
    bottom += vals

ax.set_yticks(y_indices[::-1])
ax.set_yticklabels(y_labels, fontsize=9)
ax.set_xlabel("Overall Route Score (Sum of Weighted Criteria Points)", fontsize=11, fontweight='bold')
ax.set_title("Explainable Feature Contributions Across Performance Tiers", fontsize=12, fontweight='bold', pad=12)
ax.axhline(len(sample_routes) - 8.5, color='gray', linestyle=':', linewidth=1.5)
ax.axhline(len(sample_routes) - 12.5, color='gray', linestyle=':', linewidth=1.5)
ax.text(82, len(sample_routes) - 4, "High-Frequency Trunk", fontsize=9, fontweight='bold', color='#1e7e34')
ax.text(82, len(sample_routes) - 10.5, "Median Feeder", fontsize=9, fontweight='bold', color='#c2591e')
ax.text(82, len(sample_routes) - 14.5, "Short Cut-Trip", fontsize=9, fontweight='bold', color='#a31d1d')
ax.legend(loc='lower right', bbox_to_anchor=(1.0, 0.05), fontsize=9, framealpha=0.95)

fig.tight_layout()
fig_contrib_path = figures_dir / 'fig_ranking_feature_contributions.png'
plt.savefig(fig_contrib_path, dpi=200)
plt.close()
print(f"Saved feature contributions figure to: {fig_contrib_path}")


Saved ranking overview figure to: C:\Users\kavyasri\Downloads\mtc-gtfs\reports\figures\fig_route_ranking.png
Saved feature contributions figure to: C:\Users\kavyasri\Downloads\mtc-gtfs\reports\figures\fig_ranking_feature_contributions.png


In [7]:
# 6. Inspection of Top Ranked and Lower Ranked Routes
print("=== Top 10 Ranked Chennai MTC Routes ===")
print(active_df[['rank', 'route_id', 'route_short_name', 'canonical_route', 'overall_score', 
                 'daily_scheduled_trips', 'effective_headway_min', 'stop_count', 'route_length_km']].head(10).to_string(index=False))

print("\n=== Lowest 5 Ranked Active Routes ===")
print(active_df[['rank', 'route_id', 'route_short_name', 'canonical_route', 'overall_score', 
                 'daily_scheduled_trips', 'effective_headway_min', 'stop_count', 'route_length_km']].tail(5).to_string(index=False))

print("\n=== Sample Automated Natural-Language Explanations ===")
for r_idx in [0, 1, 10, len(active_df)//2, len(active_df)-1]:
    r = active_df.iloc[r_idx]
    print(f"\n[Route {r['route_id']} - Line {r['route_short_name']}]:")
    print(f"  {r['score_explanation']}")


=== Top 10 Ranked Chennai MTC Routes ===
 rank route_id route_short_name canonical_route  overall_score  daily_scheduled_trips  effective_headway_min  stop_count  route_length_km
    1    24072               91              91          77.84                  202.0                    5.0          36        28.019984
    2    24073             91 R              91          77.79                  202.0                    5.0          35        27.741746
    3    20163             25 R              25          76.93                  210.0                    5.0          33        23.307055
    4    20089               62              62          76.37                  178.0                    5.0          48        29.586151
    5    20167            25 R1              25          76.31                  208.0                    5.0          37        22.497580
    6    20088               62              62          76.00                  171.0                    5.0          49        28.

In [8]:
# 7. Verification and Mathematical Integrity Validation
print("=== Validation Suite ===")

# 1. Total routes check
assert len(full_results_df) == 4611, f"Expected 4611 routes, got {len(full_results_df)}"
print(f"  [PASS] Total route records validated: {len(full_results_df):,}")

# 2. Active routes check
assert len(active_df) == 3934, f"Expected 3934 active routes, got {len(active_df)}"
print(f"  [PASS] Active ranked routes validated: {len(active_df):,}")

# 3. Unserviced routes check
assert len(unserviced_df) == 677, f"Expected 677 unserviced routes, got {len(unserviced_df)}"
print(f"  [PASS] Unserviced quarantined routes validated: {len(unserviced_df):,}")

# 4. Duplicate route_id check
assert full_results_df['route_id'].duplicated().sum() == 0, "Duplicate route_ids detected!"
print(f"  [PASS] Zero duplicate route_ids verified.")

# 5. Score bounds check
assert (active_df['overall_score'] >= 0.0).all() and (active_df['overall_score'] <= 100.0).all(), "Scores out of [0, 100] bounds!"
print(f"  [PASS] Score bounds verified: [{active_df['overall_score'].min():.2f}, {active_df['overall_score'].max():.2f}]")

# 6. Additive consistency check (sum of contributions == overall score)
residuals = np.abs(active_df[contrib_cols].sum(axis=1) - active_df['overall_score'])
assert (residuals < 0.05).all(), "Contribution sum does not match overall score!"
print(f"  [PASS] Additive explainability verified (max residual: {residuals.max():.4f} pts).")

# 7. Strict rank monotonicity check
assert (active_df['overall_score'].diff().dropna() <= 0).all(), "Rank order is not monotonically decreasing!"
print(f"  [PASS] Monotonic rank ordering verified.")

print("\nALL VALIDATION CHECKS PASSED SUCCESSFULLY.")


=== Validation Suite ===
  [PASS] Total route records validated: 4,611
  [PASS] Active ranked routes validated: 3,934
  [PASS] Unserviced quarantined routes validated: 677
  [PASS] Zero duplicate route_ids verified.
  [PASS] Score bounds verified: [2.13, 77.84]
  [PASS] Additive explainability verified (max residual: 0.0050 pts).
  [PASS] Monotonic rank ordering verified.

ALL VALIDATION CHECKS PASSED SUCCESSFULLY.


## 4. Academic & Operational Integrity Statement

> **Methodological Guardrails & Limitations:**
> 1. **Supply-Side Metric:** This ranking evaluates scheduled transit service supply, timetable frequency, and transit network reach. It **does not represent passenger demand, ridership volumes, or financial farebox revenue**, as empirical passenger boardings are unavailable in GTFS.
> 2. **Operational Function:** A lower score does not signify that a route is 'bad' or should be eliminated. Many lower-scoring routes are designated school cut-trips, midnight pull-outs, or specialized suburban feeder services operating a single targeted trip.
> 3. **Reproducibility:** All rankings are generated deterministically from GTFS tables without stochastic perturbation or random seeding.
